# Import and setup

In [ ]:
%load_ext autoreload
%matplotlib inline
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd
from scipy.optimize import curve_fit

# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '192.168.0.103'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

In [ ]:
default_config = qd.NVConfiguration()

# ADC channel: 0 or 1 (run diagnostic above if unsure which has your photodiode)
# 0 for ADC_D and 1 for ADC_C
default_config.adc_channel = 0
default_config.mw_channel = 1
default_config.mw_nqz = 1
default_config.mw_gain = 5000
default_config.laser_gate_pmod = 0
default_config.relax_delay_tns = 500
default_config.readout_integration_tus = 213  # max ~106.7 µs; auto-sets _treg and _tns

print("Default configuration:")
print(f"  ADC channel:     {default_config.adc_channel}")
print(f"  MW channel:      {default_config.mw_channel}")
print(f"  MW gain:         {default_config.mw_gain}")
print(f"  Relax delay:     {default_config.relax_delay_tns:.0f} ns")
print(f"  Readout int.:    {default_config.readout_integration_tus:.2f} µs  ({default_config.readout_integration_treg} treg)")

# PL Readout

## Both Channel Live

In [ ]:
from qickdawg.nvpulsing.nv_live_helpers import run_fast_pl_no_display, run_live_odmr

# Live PL — original single-point acquisition.
# Each call to PLIntensity.acquire() returns one averaged value per channel (reps=cfg.reps).
# ~165 ms per displayed point. Stop with Interrupt (■).

from collections import deque
from time import perf_counter
from IPython.display import display as _ipy_display

cfg_c = copy(default_config); cfg_c.adc_channel = 1  # ADC_C
cfg_d = copy(default_config); cfg_d.adc_channel = 0  # ADC_D
prog_c = qd.PLIntensity(cfg_c)
prog_d = qd.PLIntensity(cfg_d)

n_pts = 400
t_c, d_c = deque(maxlen=n_pts), deque(maxlen=n_pts)
t_d, d_d = deque(maxlen=n_pts), deque(maxlen=n_pts)

plt.ion()
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle('Live PL — Channel C (ADC_C=1) and Channel D (ADC_D=0)')
line_c, = axes[0].plot([], [], 'C0')
line_d, = axes[1].plot([], [], 'C1')
for ax in axes:
    ax.set_ylabel('PL (ADC units)'); ax.grid(True, alpha=0.3)
axes[1].set_xlabel('Time (s)')
handle = _ipy_display(fig, display_id=True)

t0 = perf_counter()
try:
    while True:
        t = perf_counter() - t0
        val_c = prog_c.acquire()  # PLIntensity.acquire() — averages reps, returns one float
        val_d = prog_d.acquire()
        t_c.append(t); d_c.append(val_c)
        t_d.append(t); d_d.append(val_d)
        line_c.set_data(list(t_c), list(d_c))
        line_d.set_data(list(t_d), list(d_d))
        axes[0].set_title('Channel C (ADC_C = 1)'); axes[0].relim(); axes[0].autoscale_view()
        axes[1].set_title('Channel D (ADC_D = 0)'); axes[1].relim(); axes[1].autoscale_view()
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        fig.canvas.draw_idle()
        handle.update(fig)
except KeyboardInterrupt:
    print('Live PL stopped.')
finally:
    plt.ioff(); plt.close(fig)


## Fast PL Readout

In [ ]:
# PL acquisition — hardware-rate, no display overhead.
# Runs for duration_sec seconds, then saves to CSV.
# Stop early with Interrupt (■).

fast_pl_filename = "fast_PL_dual_channel.csv"
df_fast_pl = run_fast_pl_no_display(
    default_config,
    duration_sec=120,   # 2 minutes; set None to run until interrupted
    batch_size=5000,
    csv_filename=fast_pl_filename,
)
n = len(df_fast_pl)
t_total = float(df_fast_pl["time_s"].iloc[-1]) if n else 0
print(f"Collected {n} points in {t_total:.2f} s  →  {n/t_total:.0f} pts/s" if t_total > 0 else f"{n} points")
